# Vertex AI Search — Retrieval Tests

Test the retrieval pipeline: engine config, basic search, extractive answers,
doc_scope filtering, and the high-level `retrieve_context` function.

**Prerequisites:**
- `.env.develop` configured with `GCP_PROJECT_ID`, `DATA_STORE_ID`, `SEARCH_ENGINE_ID`
- Search engine upgraded to Enterprise tier
- 225 PDFs imported to datastore

In [1]:
# Cell 1: Setup — add project root to path
import sys
from pathlib import Path

# Ensure project root is on path for src imports
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

Project root: /home/sakana/Code/ManifoldsTeam/AIServices


In [2]:
# Cell 2: Verify settings load correctly
from src.config import get_settings

settings = get_settings()
print(f"GCP Project:      {settings.gcp_project_id}")
print(f"Data Store ID:    {settings.data_store_id}")
print(f"Search Engine ID: {settings.search_engine_id}")
print(f"Location:         {settings.data_store_location}")
print(f"Engine path:      {settings.search_engine_path}")

GCP Project:      green-mercury-485016-n1
Data Store ID:    aiservice-datastore-m1_1772802306291
Search Engine ID: gp-mathagent_1773042630372
Location:         global
Engine path:      projects/green-mercury-485016-n1/locations/global/collections/default_collection/engines/gp-mathagent_1773042630372


In [3]:
# Cell 3: Check engine edition (should be SEARCH_TIER_ENTERPRISE = 2)
from google.cloud import discoveryengine_v1

engine_client = discoveryengine_v1.EngineServiceClient()
engine_name = f"projects/{settings.gcp_project_id}/locations/global/collections/default_collection/engines/{settings.search_engine_id}"
engine = engine_client.get_engine(name=engine_name)

print(f"Engine: {engine.display_name}")
print(f"Search tier: {engine.search_engine_config.search_tier} (2=Enterprise)")
print(f"Add-ons: {engine.search_engine_config.search_add_ons}")
print(f"Data stores: {engine.data_store_ids}")

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Engine: GP-mathagent
Search tier: 2 (2=Enterprise)
Add-ons: [<SearchAddOn.SEARCH_ADD_ON_LLM: 1>]
Data stores: ['aiservice-datastore-m1_1772802306291']


## 1. Raw API — Basic Search (no enterprise features)

In [4]:
# Cell 4: Raw API basic search (no enterprise features)
search_client = discoveryengine_v1.SearchServiceClient()

serving_config = f"{settings.search_engine_path}/servingConfigs/default_config"
print(f"Serving config: {serving_config}")

request = discoveryengine_v1.SearchRequest(
    serving_config=serving_config,
    query="phương trình bậc hai toán lớp 10",
    page_size=5,
)

response = search_client.search(request)
print(f"\nTotal results: {response.total_size}")
for i, result in enumerate(response.results):
    doc = result.document
    title = doc.derived_struct_data.get("title", "??") if doc.derived_struct_data else "??"
    link = doc.derived_struct_data.get("link", "") if doc.derived_struct_data else ""
    print(f"  [{i}] {title}")
    print(f"      link: {link}")

Serving config: projects/green-mercury-485016-n1/locations/global/collections/default_collection/engines/gp-mathagent_1773042630372/servingConfigs/default_config

Total results: 112
  [0] lesson_plan_user
      link: gs://documents-development-bucket/users/test-user/2026-03-07/test/lesson_plan_user.txt
  [1] Tin học 10
      link: gs://documents-development-bucket/system/sgk/lop-10/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf
  [2] sach-giao-khoa-vat-li-10-nang-cao
      link: gs://documents-development-bucket/system/sgk/lop-10/vat-li/sach-giao-khoa-vat-li-10-nang-cao.pdf
  [3] sach-giao-khoa-hinh-hoc-10-nang-cao
      link: gs://documents-development-bucket/system/sgk/lop-10/toan/sach-giao-khoa-hinh-hoc-10-nang-cao.pdf
  [4] Hinh hoc 10 2017
      link: gs://documents-development-bucket/system/sgk/lop-10/toan/sach-giao-khoa-hinh-hoc-10.pdf


## 2. Raw API — Search with Extractive Answers (enterprise)

In [5]:
# Cell 5: Raw API with extractive answers (requires Enterprise edition)
request_ea = discoveryengine_v1.SearchRequest(
    serving_config=serving_config,
    query="phương trình bậc hai toán lớp 10",
    page_size=3,
    content_search_spec=discoveryengine_v1.SearchRequest.ContentSearchSpec(
        extractive_content_spec=discoveryengine_v1.SearchRequest.ContentSearchSpec.ExtractiveContentSpec(
            max_extractive_answer_count=3,
        )
    ),
)

try:
    response_ea = search_client.search(request_ea)
    print(f"Total results: {response_ea.total_size}")
    for i, result in enumerate(response_ea.results):
        doc = result.document
        title = doc.derived_struct_data.get("title", "??") if doc.derived_struct_data else "??"
        eas = doc.derived_struct_data.get("extractive_answers", []) if doc.derived_struct_data else []
        print(f"  [{i}] {title} — {len(eas)} extractive answer(s)")
        for j, ea in enumerate(eas):
            content = str(ea.get("content", ""))[:200]
            print(f"      EA[{j}]: {content}...")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    print("\n⚠️ Enterprise tier may need propagation time (up to a few hours after enabling).")

Total results: 112
  [0] lesson_plan_user — 1 extractive answer(s)
      EA[0]: # Kế hoạch bài giảng - <b>Phương trình bậc hai</b> ## Mục tiêu bài học - Học sinh hiểu công thức nghiệm <b>phương trình bậc hai</b> - Vận dụng được công thức vào giải bài tập ## Nội dung chính ### <b>...
  [1] Tin học 10 — 2 extractive answer(s)
      EA[0]: 5. ViÕt tµi liÖu Tµi liÖu ph¶im« t¶ bµi to¸n, thuËt to¸n, thiÕt kÕ ch¬ng tr×nh, kÕt qu¶ thö nghiÖm vµ híng dÉn sö dông....
      EA[1]: Muèn nhËn biÕt mét ®èi tîng nµo ®ã, ta ph¶i biÕt ®ñ lîng th«ng tin vÒ nã....
  [2] sach-giao-khoa-vat-li-10-nang-cao — 3 extractive answer(s)
      EA[0]: d) Ph©n tÝch mét lùc thµnh <b>hai</b> lùc song song Ph©n tÝch mét lùc thµnh <b>hai</b> lùc vµ song song víi tøc lµ t×m <b>hai</b> lùc vµ song song vµ cã hîp lùc lµ . Cã v« sè c¸ch ph©n tÝch mét lùc ®·...
      EA[1]: 3 2 22 lxx a a a l =−= − = = ττ τ 2 2 1 (2 ) 2 x a = τ 2 1 <b>10</b> 1 1 0 2 lxx x a = − = −= τ 2 1 1 2 x a = τ 1 2 . 2 x at = 35....
      EA[2]: <b>Hái

## 3. LangChain Retriever — via `vertex_search.py` service

In [6]:
# Cell 6: Test get_retriever with system scope (no filter — all 225 system PDFs)
from src.services.vertex_search import get_retriever, retrieve_with_user_first

# Use get_extractive_answers=False to avoid Enterprise feature issues during propagation
retriever = get_retriever(
    user_id="__system__",
    doc_scope="system",
    max_documents=5,
    get_extractive_answers=False,
)

print(f"Serving config: {retriever._serving_config}")
print(f"Engine path in config: ✅" if "engines/" in retriever._serving_config else "❌ MISSING engine path")

2026-03-10 09:35:59 [debug    ] creating_retriever             doc_scope=system extractive_answers=False filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
Serving config: projects/green-mercury-485016-n1/locations/global/collections/default_collection/engines/gp-mathagent_1773042630372/servingConfigs/default_config
Engine path in config: ✅


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


In [7]:
# Cell 7: Invoke retriever
query = "phương trình bậc hai toán lớp 10"
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents for: '{query}'")
for i, doc in enumerate(docs):
    print(f"\n--- Doc [{i}] ---")
    print(f"Content length: {len(doc.page_content)} chars")
    print(f"Content preview: {doc.page_content[:300]}...")
    print(f"Metadata: {doc.metadata}")

Retrieved 5 documents for: 'phương trình bậc hai toán lớp 10'

--- Doc [0] ---
Content length: 705 chars
Content preview: # Kế hoạch bài giảng - Phương trình bậc hai

## Mục tiêu bài học
- Học sinh hiểu công thức nghiệm phương trình bậc hai
- Vận dụng được công thức vào giải bài tập

## Nội dung chính

### Phương trình bậc hai tổng quát
ax² + bx + c = 0 (a ≠ 0)

### Công thức nghiệm
Δ = b² - 4ac

- Nếu Δ > 0: Phương tr...
Metadata: {'id': '1cf732e1f33a6c3cc0980cee3688500c', 'source': 'gs://documents-development-bucket/users/test-user/2026-03-07/test/lesson_plan_user.txt', 'previous_segments': [], 'next_segments': []}

--- Doc [1] ---
Content length: 822 chars
Content preview: 92

Ph−¬ng tr×nh x2  y

2

 2ax  2by  c  0, víi ®iÒu kiÖn

a
2

 b2 > c, lµ ph−¬ng tr×nh cña ®−êng trßn t©m I(



a ;



b),

b¸n kÝnh R 

 
2

2
abc.

2
Khi a
2
 b
2

 c, h·y t×m tËp hîp c¸c ®iÓm M cã to¹ ®é (x ; y) tho¶ m·n ph−¬ng tr×nh (2).
? Trong c¸c ph−¬ng tr×nh sau, ph−¬ng tr×nh ...
Metadata: {

In [8]:
# Cell 8: Test with extractive answers (may fail if enterprise not yet propagated)
try:
    retriever_ea = get_retriever(
        user_id="__system__",
        doc_scope="system",
        max_documents=3,
        get_extractive_answers=True,
    )
    docs_ea = retriever_ea.invoke(query)
    print(f"✅ Extractive answers work! Retrieved {len(docs_ea)} docs")
    for i, doc in enumerate(docs_ea):
        print(f"  [{i}] {len(doc.page_content)} chars: {doc.page_content[:150]}...")
except Exception as e:
    print(f"❌ Extractive answers not yet available: {type(e).__name__}")
    print(f"   Enterprise propagation may still be in progress.")

2026-03-10 09:47:57 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


✅ Extractive answers work! Retrieved 6 docs
  [0] 355 chars: # Kế hoạch bài giảng - <b>Phương trình bậc hai</b> ## Mục tiêu bài học - Học sinh hiểu công thức nghiệm <b>phương trình bậc hai</b> - Vận dụng được cô...
  [1] 119 chars: 5. ViÕt tµi liÖu Tµi liÖu ph¶im« t¶ bµi to¸n, thuËt to¸n, thiÕt kÕ ch¬ng tr×nh, kÕt qu¶ thö nghiÖm vµ híng dÉn sö dông....
  [2] 73 chars: Muèn nhËn biÕt mét ®èi tîng nµo ®ã, ta ph¶i biÕt ®ñ lîng th«ng tin vÒ nã....
  [3] 205 chars: d) Ph©n tÝch mét lùc thµnh <b>hai</b> lùc song song Ph©n tÝch mét lùc thµnh <b>hai</b> lùc vµ song song víi tøc lµ t×m <b>hai</b> lùc vµ song song vµ ...
  [4] 130 chars: 3 2 22 lxx a a a l =−= − = = ττ τ 2 2 1 (2 ) 2 x a = τ 2 1 <b>10</b> 1 1 0 2 lxx x a = − = −= τ 2 1 1 2 x a = τ 1 2 . 2 x at = 35....
  [5] 51 chars: <b>Hái</b> vÞ trÝ ®ã c¸ch n¬i xuÊt ph¸t bao xa ? 6....


## 4. High-level `retrieve_context` (with fallback)

In [9]:
# Cell 9: Test retrieve_context (async) — has built-in fallback
import asyncio
from src.services.vertex_search import retrieve_context

context_strings, source_metadata = await retrieve_context(
    user_id="__system__",
    query="giải phương trình bậc hai toán lớp 10",
    doc_scope="system",
    max_documents=5,
)

print(f"Context chunks: {len(context_strings)}")
print(f"Sources: {len(source_metadata)}")
for i, (ctx, meta) in enumerate(zip(context_strings, source_metadata)):
    print(f"\n--- Chunk [{i}] ---")
    print(f"Source: {meta.get('source', '??')}")
    print(f"Scope: {meta.get('doc_scope', '??')}")
    print(f"Content ({len(ctx)} chars): {ctx[:200]}...")

2026-03-10 09:48:09 [info     ] retrieving_context             doc_scope=system query='giải phương trình bậc hai toán lớp 10' user_id=__system__
2026-03-10 09:48:09 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:09 [debug    ] retrieving_documents           doc_scope=system query='giải phương trình bậc hai toán lớp 10'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 09:48:11 [debug    ] retrieved_documents            count=12
2026-03-10 09:48:11 [info     ] context_retrieved              chunks=12 sources=12
Context chunks: 12
Sources: 12

--- Chunk [0] ---
Source: gs://documents-development-bucket/users/test-user/2026-03-07/test/lesson_plan_user.txt
Scope: user
Content (215 chars): <b>Giải phương trình</b>: x² - 5x + 6 = 0 Δ = 25 - 24 = 1 &gt; 0 x₁ = (5 + 1)/2 = 3 x₂ = (5 - 1)/2 = 2 2. <b>Giải phương trình</b>: 2x² + 3x - 5 = 0 Δ = 9 + 40 = 49 &gt; 0 x₁ = (-3 + 7)/4 = 1 x₂ = (-3...

--- Chunk [1] ---
Source: gs://documents-development-bucket/system/sgk/lop-10/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf50
Scope: user
Content (119 chars): 5. ViÕt tµi liÖu Tµi liÖu ph¶im« t¶ bµi to¸n, thuËt to¸n, thiÕt kÕ ch¬ng tr×nh, kÕt qu¶ thö nghiÖm vµ híng dÉn sö dông....

--- Chunk [2] ---
Source: gs://documents-development-bucket/system/sgk/lop-10/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf7
Scope: user
Content (158 chars): VÝ dô, xÐt viÖc tung ngÉu nhiªn ®ång

## 5. Multi-query Test — Different Subjects

In [10]:
# Cell 10: Test multiple queries across different subjects
test_queries = [
    "phương trình bậc hai toán lớp 10",
    "cấu tạo nguyên tử hóa học lớp 10",
    "lịch sử Việt Nam thời kỳ Bắc thuộc lớp 10",
    "hệ sinh thái sinh học lớp 12",
    "present perfect tense English grade 11",
]

for q in test_queries:
    ctx, meta = await retrieve_context(
        user_id="__system__",
        query=q,
        doc_scope="system",
        max_documents=3,
    )
    sources = [m.get("source", "?")[-40:] for m in meta]
    print(f"Query: {q[:50]:50s} → {len(ctx)} chunks | sources: {sources}")

2026-03-10 09:48:26 [info     ] retrieving_context             doc_scope=system query='phương trình bậc hai toán lớp 10' user_id=__system__
2026-03-10 09:48:26 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:26 [debug    ] retrieving_documents           doc_scope=system query='phương trình bậc hai toán lớp 10'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 09:48:28 [debug    ] retrieved_documents            count=6
2026-03-10 09:48:28 [info     ] context_retrieved              chunks=6 sources=6
Query: phương trình bậc hai toán lớp 10                   → 6 chunks | sources: ['ser/2026-03-07/test/lesson_plan_user.txt', '/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf50', '0/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf7', 'sach-giao-khoa-vat-li-10-nang-cao.pdf130', '/sach-giao-khoa-vat-li-10-nang-cao.pdf36', '/sach-giao-khoa-vat-li-10-nang-cao.pdf18']
2026-03-10 09:48:28 [info     ] retrieving_context             doc_scope=system query='cấu tạo nguyên tử hóa học lớp 10' user_id=__system__
2026-03-10 09:48:28 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:28 [debug    ] retrieving_documents           doc_scope=system query='cấu tạo nguyên tử hóa học lớp 10'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 09:48:30 [debug    ] retrieved_documents            count=8
2026-03-10 09:48:30 [info     ] context_retrieved              chunks=8 sources=8
Query: cấu tạo nguyên tử hóa học lớp 10                   → 8 chunks | sources: ['-khoa-vat-li-11-chan-troi-sang-tao.pdf72', '-khoa-vat-li-11-chan-troi-sang-tao.pdf74', 'khoa-vat-li-11-chan-troi-sang-tao.pdf104', 'sach-giao-khoa-hoa-hoc-10-nang-cao.pdf24', 'sach-giao-khoa-hoa-hoc-10-nang-cao.pdf51', 'ach-giao-khoa-hoa-hoc-10-nang-cao.pdf117', 'tin-hoc/sach-giao-khoa-tin-hoc-10.pdf167', '/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf89']
2026-03-10 09:48:30 [info     ] retrieving_context             doc_scope=system query='lịch sử Việt Nam thời kỳ Bắc thuộc lớp 10' user_id=__system__
2026-03-10 09:48:30 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:30 [debug    ] retrieving_documents          

...

2026-03-10 09:48:32 [debug    ] retrieved_documents            count=3
2026-03-10 09:48:32 [info     ] context_retrieved              chunks=3 sources=3
Query: lịch sử Việt Nam thời kỳ Bắc thuộc lớp 10          → 3 chunks | sources: ['en-de-hoc-tap-lich-su-10-canh-dieu.pdf58', 'lich-su/sach-giao-khoa-lich-su-10.pdf206', '-10-ket-noi-tri-thuc-voi-cuoc-song.pdf66']
2026-03-10 09:48:32 [info     ] retrieving_context             doc_scope=system query='hệ sinh thái sinh học lớp 12' user_id=__system__
2026-03-10 09:48:32 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:32 [debug    ] retrieving_documents           doc_scope=system query='hệ sinh thái sinh học lớp 12'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 09:48:33 [debug    ] retrieved_documents            count=8
2026-03-10 09:48:33 [info     ] context_retrieved              chunks=8 sources=8
Query: hệ sinh thái sinh học lớp 12                       → 8 chunks | sources: ['ch-giao-khoa-sinh-hoc-11-nang-cao.pdf150', 'ach-giao-khoa-sinh-hoc-11-nang-cao.pdf52', 'ach-giao-khoa-sinh-hoc-11-nang-cao.pdf53', 'ach-giao-khoa-sinh-hoc-10-nang-cao.pdf10', 'ch-giao-khoa-sinh-hoc-10-nang-cao.pdf113', 'ach-giao-khoa-sinh-hoc-10-nang-cao.pdf13', 'yen-de-hoc-tap-dia-li-12-canh-dieu.pdf34', 'yen-de-hoc-tap-dia-li-12-canh-dieu.pdf60']
2026-03-10 09:48:33 [info     ] retrieving_context             doc_scope=system query='present perfect tense English grade 11' user_id=__system__
2026-03-10 09:48:33 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 09:48:33 [debug    ] retrieving_documents           do

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 09:48:35 [debug    ] retrieved_documents            count=9
2026-03-10 09:48:35 [info     ] context_retrieved              chunks=9 sources=9
Query: present perfect tense English grade 11             → 9 chunks | sources: ['h/sach-hoc-sinh-tieng-anh-11-think.pdf48', '/sach-hoc-sinh-tieng-anh-11-think.pdf107', 'h/sach-hoc-sinh-tieng-anh-11-think.pdf51', 'ng-anh/sach-giao-khoa-tieng-anh-11.pdf57', 'eng-anh/sach-giao-khoa-tieng-anh-11.pdf9', 'ng-anh/sach-giao-khoa-tieng-anh-11.pdf31', 'ng-anh/sach-giao-khoa-tieng-anh-10.pdf62', 'ng-anh/sach-giao-khoa-tieng-anh-10.pdf82', 'ng-anh/sach-giao-khoa-tieng-anh-10.pdf81']
